# MedLook-4B — Colab Train + Eval Notebook

> **Research prototype only. Not for clinical use, diagnosis, or treatment decisions.**

This notebook is the **only** place real training/evaluation happens. Local machines
are for pipeline validation only (`--dry-run`, `--mock`) — never for the real 4B model.

## Goal: prove Full-MedLook improves over what exists today

Compare three systems on the **same held-out eval** (no human raters required):

| System | Meaning |
|---|---|
| **Base** | MedGemma-4B, no fine-tune (today's starting point) |
| **Short-SFT** | Answer-only SFT (standard fine-tune technique) |
| **Full-MedLook** | STRATEGY + PROCESS + FINAL (our method) |

**Success gate:** Full must beat Base **and** Short on answer F1 (≥1 primary set) **and**
on calibration (lower AURC **or** lower overconfident-error). Read `success_gate` honestly.

## Fast-compare protocol (Colab Pro)

Use **A100** if available (Runtime → Change runtime type). L4 works but is slower; T4 is tight.

Default knobs in the config cell (`FAST_COMPARE = True`):
- Matched `max_steps: 300` for Short and Full (fair ablation, ~half of 1.5-epoch runs)
- `max_seq_length: 2048`, batch 2 / accum 4 when VRAM allows
- Auto-resume Short from the latest Drive checkpoint
- Skip Process-SFT unless you need the 4-system table

## Order of cells

1. GPU + Drive + clone + deps + HF login
2. Gold set + prepare Short + Full data to Drive
3. Packing smoke test
4. Train Short (resume) → Full (matched steps)
5. Predictions for base / short_sft / full_medlook → `eval.py`
6. Diagnose failures (what to improve next)
7. Optional: Process-SFT, export, Gradio

Do **not** re-seed hoping for a pass. If the gate fails, use the diagnosis cell.

## 1. Check GPU, mount Drive, clone the repo, install dependencies

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Official MedLook-4B repo
REPO_URL = "https://github.com/Shaked-g/unsloth2.git"
REPO_DIR = "/content/unsloth2"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only || true

%cd {REPO_DIR}

In [ ]:
%pip install -q -r requirements.txt -r requirements-colab.txt
%pip install -q -e . --no-deps

In [ ]:
import getpass

from huggingface_hub import login

# Prefer Colab secret / env var. Never commit a token into this notebook.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata

        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if not hf_token:
    hf_token = getpass.getpass(
        "Hugging Face token (MedGemma license must be accepted): "
    )
login(hf_token)
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print("HF login OK")

## 2. Build gold set + prepare MVP training data (~3.5k mixed samples)

Data mix: VQA-RAD + PathVQA + SLAKE (`hf_limit` 700), optional Meissa (graceful off),
uncertainty anti-shortcut, `target_size: 3500`, held-out auto gold set (~200 cases).

The next cell rewrites configs to Drive **and** applies `FAST_COMPARE` train knobs so
Short and Full use the **same step budget**. Set `FAST_COMPARE = False` only if you
intentionally want the slower 1.5-epoch YAML defaults.

Skip re-preparing if Drive already has `medlook_runs/data/{short_sft,full_medlook}/train.jsonl`.

In [ ]:
import glob
import yaml

DRIVE_ROOT = "/content/drive/MyDrive/medlook_runs"
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Fast matched ablations for Colab Pro. Flip to False only for full 1.5-epoch runs.
FAST_COMPARE = True
MAX_STEPS = 300          # same budget for Short and Full (fair comparison)
MAX_SEQ_LENGTH = 2048    # faster than 4096; raise only if PROCESS/RELOOK truncates
TRAIN_BATCH = 2          # try 4 on A100; drop to 1 on OOM
GRAD_ACCUM = 4           # effective batch ~= 8 (same as YAML default 1x8)
SAVE_STEPS = 50          # denser checkpoints for resume after disconnects


def make_colab_config(src_path: str, dst_path: str) -> str:
    with open(src_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    name = cfg["name"]
    cfg["data"]["output_dir"] = f"{DRIVE_ROOT}/data/{name}"
    cfg["train"]["output_dir"] = f"{DRIVE_ROOT}/checkpoints/{name}"
    if FAST_COMPARE:
        cfg["model"]["max_seq_length"] = MAX_SEQ_LENGTH
        cfg["train"]["num_train_epochs"] = None
        cfg["train"]["max_steps"] = MAX_STEPS
        cfg["train"]["per_device_train_batch_size"] = TRAIN_BATCH
        cfg["train"]["gradient_accumulation_steps"] = GRAD_ACCUM
        cfg["train"]["save_steps"] = SAVE_STEPS
        cfg["train"]["save_total_limit"] = 6
    with open(dst_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst_path


SHORT_SFT_CFG = make_colab_config("configs/short_sft.yaml", "configs/short_sft.colab.yaml")
PROCESS_SFT_CFG = make_colab_config("configs/process_sft.yaml", "configs/process_sft.colab.yaml")
FULL_MEDLOOK_CFG = make_colab_config("configs/full_medlook.yaml", "configs/full_medlook.colab.yaml")

print("FAST_COMPARE =", FAST_COMPARE, "| max_steps =", MAX_STEPS if FAST_COMPARE else "epochs from YAML")
print(SHORT_SFT_CFG)
print(FULL_MEDLOOK_CFG)


def latest_checkpoint(run_name: str):
    root = f"{DRIVE_ROOT}/checkpoints/{run_name}"
    ckpts = glob.glob(f"{root}/checkpoint-*")
    if not ckpts:
        return None

    def _step(p: str) -> int:
        try:
            return int(p.rsplit("-", 1)[-1])
        except ValueError:
            return -1

    return max(ckpts, key=_step)


SHORT_RESUME = latest_checkpoint("short_sft")
FULL_RESUME = latest_checkpoint("full_medlook")
print("Short resume:", SHORT_RESUME or "(none — fresh start)")
print("Full resume:", FULL_RESUME or "(none — fresh start)")

In [ ]:
# 1) Held-out gold strategy set (~200 cases). Rebuild if JSON or images missing.
_gold_json = "data/gold_strategy_set.json"
_gold_img_dir = "data/gold_strategy_set_images"
_gold_imgs_ok = (
    os.path.isdir(_gold_img_dir)
    and len([f for f in os.listdir(_gold_img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]) > 0
)
if not (os.path.exists(_gold_json) and _gold_imgs_ok):
    print("Building gold strategy set (JSON and/or images missing)...")
    !python scripts/build_gold_strategy_set.py --per-action 50 \
        --out-json {_gold_json} \
        --out-image-dir {_gold_img_dir}
else:
    print(f"Gold strategy set OK ({len(os.listdir(_gold_img_dir))} image files).")

# 2) Prepare Short + Full only (Process is optional later).
# Skip a profile if its train.jsonl already exists on Drive.
PREPARE_PROCESS = False  # set True only for the 4-system table


def _need_prepare(cfg_path: str) -> bool:
    with open(cfg_path, "r", encoding="utf-8") as f:
        out = yaml.safe_load(f)["data"]["output_dir"]
    train_jsonl = os.path.join(out, "train.jsonl")
    ok = os.path.exists(train_jsonl)
    print(f"{'SKIP' if ok else 'PREPARE'} {train_jsonl}")
    return not ok


if _need_prepare(SHORT_SFT_CFG):
    !python scripts/prepare_data.py --config {SHORT_SFT_CFG}
if _need_prepare(FULL_MEDLOOK_CFG):
    !python scripts/prepare_data.py --config {FULL_MEDLOOK_CFG}
if PREPARE_PROCESS and _need_prepare(PROCESS_SFT_CFG):
    !python scripts/prepare_data.py --config {PROCESS_SFT_CFG}

## 3. Multi-image packing smoke test (do this before any full training run)

Runs the real base model + the real `UnslothVisionDataCollator` on the first 50
prepared training samples, without starting a training loop. Multi-image packing
failures are the single most likely way to waste hours of Colab compute -- this
catches them in under a minute.

In [ ]:
!python scripts/train.py --config {FULL_MEDLOOK_CFG} --packing-smoke-test 50

## 4. Train Short-SFT → Full-MedLook (matched steps)

With `FAST_COMPARE=True`, both runs use the same `max_steps` so the comparison is fair.

1. **Short-SFT** — resume from latest Drive checkpoint if present (e.g. `checkpoint-200`).
2. **Full-MedLook** — train next (resume only if a Full checkpoint already exists).

Process-SFT is skipped by default. After training, adapters land at:
`{DRIVE_ROOT}/checkpoints/{name}/final_adapter`

If a run dies mid-way before `final_adapter` is written, re-run the same cell — resume
picks up the latest `checkpoint-*`.

In [ ]:
# Short-SFT: [FINAL] only. Auto-resumes from latest checkpoint on Drive.
SHORT_RESUME = latest_checkpoint("short_sft")
if SHORT_RESUME:
    print("Resuming Short-SFT from", SHORT_RESUME)
    !python scripts/train.py --config {SHORT_SFT_CFG} --resume-from-checkpoint {SHORT_RESUME}
else:
    print("Starting Short-SFT from scratch")
    !python scripts/train.py --config {SHORT_SFT_CFG}

In [ ]:
# Full-MedLook: [STRATEGY] + [PROCESS] + [FINAL] — same step budget as Short.
FULL_RESUME = latest_checkpoint("full_medlook")
if FULL_RESUME:
    print("Resuming Full-MedLook from", FULL_RESUME)
    !python scripts/train.py --config {FULL_MEDLOOK_CFG} --resume-from-checkpoint {FULL_RESUME}
else:
    print("Starting Full-MedLook from scratch")
    !python scripts/train.py --config {FULL_MEDLOOK_CFG}

In [ ]:
# OPTIONAL — Process-SFT ([PROCESS]+[FINAL], no STRATEGY). Skip for the main claim.
# Only run if success_gate already passed and you want the 4-system ablation table.
RUN_PROCESS_SFT = False
if RUN_PROCESS_SFT:
    PROC_RESUME = latest_checkpoint("process_sft")
    if PROC_RESUME:
        !python scripts/train.py --config {PROCESS_SFT_CFG} --resume-from-checkpoint {PROC_RESUME}
    else:
        !python scripts/train.py --config {PROCESS_SFT_CFG}
else:
    print("Skipping Process-SFT (set RUN_PROCESS_SFT = True to enable).")

## 5. Prove improvement: predictions + three-system eval

Generate Base / Short / Full on held-out `test` + auto gold strategy set, then run
`eval.py` (**never** `--mock` for claims).

This is the comparison to “what exists today”:
- Base = pretrained MedGemma (no MedLook training)
- Short-SFT = standard answer SFT
- Full-MedLook = our method

Adapters must exist at `.../final_adapter` (written when a train run finishes). If you
only have `checkpoint-*`, finish or resume that run first.

In [ ]:
PRED_DIR = f"{DRIVE_ROOT}/predictions"
EVAL_SPLIT = "test"  # held-out; must NOT be the train split used in prepare_data

assert os.path.exists("data/gold_strategy_set.json"), (
    "Gold strategy set missing — re-run the prepare cell so strategy/calibration metrics exist."
)
GOLD_ARGS = (
    "--gold-strategy-json data/gold_strategy_set.json "
    "--gold-strategy-image-dir data/gold_strategy_set_images"
)
CHECKPOINTS = f"{DRIVE_ROOT}/checkpoints"

for name in ("short_sft", "full_medlook"):
    adapter = f"{CHECKPOINTS}/{name}/final_adapter"
    print(("OK" if os.path.isdir(adapter) else "MISSING"), adapter)

In [ ]:
# Base: no adapter -- uses the plain pretrained model.
!python scripts/generate_predictions.py --config {FULL_MEDLOOK_CFG} \
    --system-name base --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}

In [ ]:
!python scripts/generate_predictions.py --config {SHORT_SFT_CFG} \
    --adapter-dir {CHECKPOINTS}/short_sft/final_adapter \
    --system-name short_sft --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}

In [ ]:
# Optional Process-SFT predictions (only if you trained it).
_run_process = globals().get("RUN_PROCESS_SFT", False)
if _run_process and os.path.isdir(f"{CHECKPOINTS}/process_sft/final_adapter"):
    !python scripts/generate_predictions.py --config {PROCESS_SFT_CFG} \
        --adapter-dir {CHECKPOINTS}/process_sft/final_adapter \
        --system-name process_sft --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}
else:
    print("Skipping process_sft predictions.")

In [ ]:
!python scripts/generate_predictions.py --config {FULL_MEDLOOK_CFG} \
    --adapter-dir {CHECKPOINTS}/full_medlook/final_adapter \
    --system-name full_medlook --split {EVAL_SPLIT} --out-dir {PRED_DIR} {GOLD_ARGS}

In [ ]:
# Real three-system report — NEVER use --mock for claims.
!python scripts/eval.py --predictions-dir {PRED_DIR} --out {DRIVE_ROOT}/report.json
print("Wrote", f"{DRIVE_ROOT}/report.json")

## 5b. Diagnose: did we improve? If not, what to fix next

Loads `report.json` and prints a concrete next experiment. Do **not** change seeds
hoping for a pass — pick one failure mode and one fix.

In [ ]:
import json
from pathlib import Path

report_path = Path(DRIVE_ROOT) / "report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
gate = report.get("success_gate", {})
systems = report.get("systems", {})

print("=== SUCCESS GATE ===")
print("passed:", gate.get("passed"))
for reason in gate.get("reasons", []):
    print(" -", reason)
print("details:", json.dumps(gate.get("details", {}), indent=2))

print("\n=== PER-SYSTEM SNAPSHOT ===")
for name, entry in systems.items():
    aq = entry.get("answer_quality") or {}
    if aq:
        n = sum(m["n"] for m in aq.values()) or 1
        f1 = sum(m["token_f1"] * m["n"] for m in aq.values()) / n
    else:
        f1 = None
    gs = entry.get("gold_strategy") or {}
    strat = gs.get("strategy") or {}
    calib = gs.get("calibration") or {}
    print(
        f"{name:14s} answer_f1={f1 if f1 is None else round(f1, 3)}  "
        f"action_f1={strat.get('macro_f1')}  "
        f"flag_p/r={strat.get('flag_precision')}/{strat.get('flag_recall')}  "
        f"aurc={calib.get('aurc')}  overconf={calib.get('overconfident_error_rate')}"
    )

# Failure → next experiment map (pick ONE; do not change everything at once).
print("\n=== WHAT TO IMPROVE NEXT (pick one) ===")
reasons = " ".join(gate.get("reasons", [])).lower()
full = systems.get("full_medlook") or {}
full_gs = (full.get("gold_strategy") or {})
full_strat = full_gs.get("strategy") or {}
cm = full_strat.get("confusion")

if gate.get("passed"):
    print("Gate PASSED. Optional next: enable Meissa if off; raise MAX_STEPS to 500; add Process-SFT ablation.")
elif "answer token-f1" in reasons:
    print(
        "Answer quality lagging vs Base/Short.\n"
        "  1) Check Meissa loaded during prepare (real RELOOK multi-image helps).\n"
        "  2) Raise MAX_STEPS to 500 with FAST_COMPARE still matched Short=Full.\n"
        "  3) If still flat: try finetune_vision_layers=true (slower) or lower LR to 1e-4.\n"
        "  4) Inspect wrong answers in predictions/*/ — truncation? schema parse fail?"
    )
elif "aurc" in reasons or "overconfident" in reasons:
    print(
        "Calibration lagging (answers may be fine).\n"
        "  1) Increase uncertainty mix weight (e.g. 0.25 → 0.35) and rebuild Full data.\n"
        "  2) Check FLAG precision/recall — if FLAG recall low, model never abstains.\n"
        "  3) Look at CONF values on wrong answers in gold predictions — should be low."
    )
else:
    print("Gate failed for other/missing metrics. Ensure base + short_sft + full_medlook predictions all exist.")

if cm:
    print("\nFull-MedLook ACTION confusion (rows=gold, cols=pred):")
    print(json.dumps(cm, indent=2))
    relook_as_answer = (cm.get("RELOOK") or {}).get("ANSWER_CONFIDENT", 0)
    flag_as_answer = (cm.get("FLAG_UNCERTAIN") or {}).get("ANSWER_CONFIDENT", 0) + (
        (cm.get("ESCALATE") or {}).get("ANSWER_CONFIDENT", 0)
    )
    if relook_as_answer > 5:
        print("TIP: Many gold RELOOK → ANSWER_CONFIDENT. Need more multi-hop/Meissa RELOOK examples.")
    if flag_as_answer > 5:
        print("TIP: Many FLAG/ESCALATE → ANSWER_CONFIDENT. Strengthen uncertainty pairs / blur severity.")

print("\nReport path:", report_path)

## 6. (Optional) Export the winning system

Merges the LoRA adapter into a standalone 16-bit model, then attempts a GGUF export
for CPU inference. GGUF export for vision-language architectures is less mature than
for text-only models -- if it fails, fall back to running llama.cpp's
`convert_hf_to_gguf.py` directly against the merged 16-bit directory (see
`medlook/export/gguf.py`'s docstring for the full caveat).

In [ ]:
from medlook.export.gguf import export_gguf
from medlook.export.merge import merge_lora_to_fp16

WINNING_CFG = FULL_MEDLOOK_CFG  # change if a different ablation actually won the eval
WINNING_ADAPTER = f"{CHECKPOINTS}/full_medlook/final_adapter"
EXPORT_DIR = f"{DRIVE_ROOT}/exports"

merged_dir = merge_lora_to_fp16(WINNING_CFG, WINNING_ADAPTER, f"{EXPORT_DIR}/merged_16bit")
print("Merged 16-bit model at:", merged_dir)

try:
    gguf_dir = export_gguf(WINNING_CFG, WINNING_ADAPTER, f"{EXPORT_DIR}/gguf", quantization="q4_k_m")
    print("GGUF export at:", gguf_dir)
except Exception as exc:
    print("GGUF export failed or is unsupported for this architecture right now:", exc)
    print("Fall back: run llama.cpp's convert_hf_to_gguf.py directly against", merged_dir)

## 7. (Optional) Launch the Gradio demo in-notebook, with real weights

In [ ]:
from medlook.demo.gradio_app import build_demo
from medlook.train.sft import load_model_with_optional_adapter

model, tokenizer = load_model_with_optional_adapter(WINNING_CFG, WINNING_ADAPTER)
demo = build_demo(model=model, tokenizer=tokenizer)
demo.launch(share=True)

## Wrap-up: honesty checklist before reporting results

- [ ] Ran `eval.py` **without** `--mock`
- [ ] Compared **Base vs Short-SFT vs Full-MedLook** with the **same** `max_steps` / data seed
- [ ] Quoted `success_gate.passed` exactly as printed
- [ ] If failed: used the diagnosis cell and chose **one** next experiment (not a seed scramble)
- [ ] Gold strategy metrics labeled as **rule-curated auto gold**, not clinician-rated
- [ ] Included 1–2 qualitative examples (RELOOK + FLAG/ESCALATE) from real predictions

**Improvement loop:** train matched Short+Full → eval → diagnosis cell → one change → retrain Full (and Short if step budget changed) → eval again.